# Módulo 2 — Sentimiento: BETO Fine-tuning
**dccuchile/bert-base-spanish-wwm-uncased** con Hugging Face Trainer

> Activar GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/smartretail360'
MODEL_DIR  = f'{DRIVE_ROOT}/models/sentiment_analyzer/beto_finetuned'

import os
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
!pip install -q transformers datasets tensorflow scikit-learn

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
from datasets import load_dataset
from transformers import AutoTokenizer
import os

MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LANG = 'es'

# Descubrir y descargar solo los JSONL del español (el repo tiene un .py incompatible con datasets >= 4.0)
all_files  = list(list_repo_files("mteb/amazon_reviews_multi", repo_type="dataset"))
es_jsonl   = sorted([f for f in all_files if f.endswith('.jsonl') and f.startswith(f'{LANG}/')])

split_files = {}
for f in es_jsonl:
    local = hf_hub_download(
        repo_id="mteb/amazon_reviews_multi", filename=f,
        repo_type="dataset", local_dir="/content/amz_reviews_es",
    )
    fname = os.path.basename(f).lower()
    if 'train' in fname:   split_files.setdefault('train', []).append(local)
    elif 'test' in fname:  split_files.setdefault('test',  []).append(local)
    elif 'val'  in fname:  split_files.setdefault('validation', []).append(local)

ds = load_dataset("json", data_files=split_files)

def rating_to_label(r):
    return 0 if r <= 2 else (1 if r == 3 else 2)

def tokenize_and_label(batch):
    out = tokenizer(batch['review_body'], truncation=True, max_length=256)
    out['labels'] = [rating_to_label(s) for s in batch['stars']]
    return out

ds = ds.map(tokenize_and_label, batched=True, remove_columns=ds['train'].column_names)
print(ds)

In [ ]:
from transformers import TFAutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
import numpy as np

model = TFAutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

tf_train = ds['train'].to_tf_dataset(
    columns=['input_ids', 'attention_mask'],
    label_cols=['labels'],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)
tf_val = ds['validation'].to_tf_dataset(
    columns=['input_ids', 'attention_mask'],
    label_cols=['labels'],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

history = model.fit(tf_train, validation_data=tf_val, epochs=3)

# Evaluación F1-macro sobre validación
all_preds, all_labels = [], []
for batch in tf_val:
    inputs, labels = batch
    logits = model(inputs, training=False).logits
    all_preds.extend(np.argmax(logits.numpy(), axis=1))
    all_labels.extend(labels.numpy())

print(f'\nF1-macro: {f1_score(all_labels, all_preds, average="macro"):.4f}')
print(classification_report(all_labels, all_preds, target_names=['negativo', 'neutro', 'positivo']))

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'BETO guardado en {MODEL_DIR}')